# PyTorch Tutorial 40: Reward Modeling and PPO for LLMs (The FAANG Standard)

In Tutorial 19, we implemented the DPO loss function — the elegant shortcut to alignment. But the **original** alignment technique, RLHF with PPO, is still used at scale by OpenAI, Anthropic, and xAI. Understanding it deeply is non-negotiable for any post-training role.

This tutorial goes beyond pseudocode. We build the **entire RLHF pipeline** from scratch with runnable code on a small transformer.

## Learning Objectives
1. **Build a Reward Model** from scratch — classifier head on a transformer backbone
2. **Implement GAE** (Generalized Advantage Estimation) — the bias-variance trick that makes PPO work
3. **Build a full PPO training loop for language models** — not CartPole, actual token-level optimization
4. **Run end-to-end RLHF** — SFT to Reward Model to PPO on a small LM
5. **Nail the FAANG interview** — reward hacking, KL penalties, scaling RLHF to 70B+

**Prerequisites**: Tutorial 13 (RL fundamentals, PPO loss), Tutorial 19 (DPO, alignment pipeline)

---

## 1. Vocabulary First

You already know the 3-stage pipeline from Tutorial 19. Now we go deeper into the RL components.

- **Reward Model (RM)**: A neural network that takes (prompt, response) and outputs a scalar score. Trained on human preference data using the Bradley-Terry model.
- **Bradley-Terry Model**: A preference model where P(A > B) = sigmoid(r(A) - r(B)). The same math as logistic regression.
- **Value Head**: An extra linear layer on top of the LLM that predicts V(s) — the expected future reward from this token position.
- **Policy Head**: The original LM head (logits to token probabilities). This is what PPO optimizes.
- **GAE (Generalized Advantage Estimation)**: A method to compute advantages that smoothly interpolates between high-bias (TD) and high-variance (Monte Carlo) estimates using a parameter lambda.
- **KL Penalty**: A term added to the reward that penalizes the policy for diverging from the reference model. Prevents reward hacking.
- **Experience Buffer**: Stores (query, response, log_probs, rewards, values) from rollouts. PPO trains on mini-batches from this buffer.
- **Reference Model**: A frozen copy of the SFT model. The KL divergence between policy and reference acts as a regularizer.

### The RLHF Pipeline (What We're Building)

```
Stage 1: SFT Model (already done, we start from a pretrained LM)
    |
    v
Stage 2: Train Reward Model
    |  Input: (prompt, response_A, response_B, human_preference)
    |  Output: R(prompt, response) -> scalar
    |  Loss: -log sigmoid(R(chosen) - R(rejected))  <- Bradley-Terry
    |
    v
Stage 3: PPO Training
    |  For each batch of prompts:
    |    1. Generate responses with Policy (LLM)
    |    2. Score with Reward Model
    |    3. Compute KL penalty against Reference Model
    |    4. Compute advantages with GAE
    |    5. PPO update (clipped objective + value loss)
    |
    v
Result: Aligned LLM
```

### Why PPO over DPO?

| Aspect | PPO (RLHF) | DPO |
|--------|-----------|-----|
| Online learning | Yes - generates fresh data | No - fixed dataset |
| Reward model | Explicit - reusable | Implicit - baked into loss |
| Compute cost | 4 models in memory | 2 models in memory |
| Stability | Harder to tune | Easier |
| Performance at scale | Often better (OpenAI, Anthropic) | Competitive |
| Reward hacking risk | Higher (need guardrails) | Lower |

**Bottom line**: DPO is simpler. PPO is more powerful (especially with online data). At frontier labs, both are used — often PPO for the final stage.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import copy
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Ready for RLHF!")

---

## 2. Part 1: Reward Model from Scratch

The Reward Model is the backbone of RLHF. It learns to predict human preferences.

**Architecture**: Take a transformer backbone, remove the LM head, add a linear layer that outputs a single scalar.

```
(prompt + response) -> Transformer backbone -> last token hidden state -> Linear(hidden, 1) -> scalar reward
```

**Why last token?** In causal (left-to-right) models, the last token's hidden state has attended to the entire sequence. It's a natural summary vector.

### FAANG Interview Question

**Q: "How does a reward model differ from a classifier?"**

**A**: A classifier predicts discrete categories (positive/negative). A reward model predicts a **relative** scalar score — it's trained on **pairs** where we only know "A is better than B", not absolute quality. The Bradley-Terry loss (log sigmoid(r_A - r_B)) means only the **difference** between rewards matters, not their absolute values. This is why reward models are fundamentally ordinal, not cardinal.

In [ ]:
class RewardModel(nn.Module):
    """
    Reward Model for RLHF.
    
    Takes a sequence of token embeddings and outputs a scalar reward.
    In production, the backbone would be a pretrained transformer.
    Here we use a small transformer for demonstration.
    
    Architecture:
        Input: [batch, seq_len] token IDs
        -> Embedding + PosEmbed -> TransformerEncoder (2 layers)
        -> Last token hidden state
        -> Linear -> scalar reward
    """
    
    def __init__(self, vocab_size=1000, embed_dim=128, n_heads=4, 
                 n_layers=2, max_seq_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, 
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        
        # Reward head: hidden state -> scalar
        self.reward_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )
    
    def forward(self, input_ids, attention_mask=None):
        """Forward pass. Returns scalar reward per sequence.
        
        Args:
            input_ids: [batch, seq_len] token indices
            attention_mask: [batch, seq_len] 1=real, 0=padding
        Returns:
            rewards: [batch] scalar reward per sequence
        """
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.transformer(x)
        
        # Extract last non-padding token's hidden state
        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
            last_hidden = x[torch.arange(batch_size), last_idx]
        else:
            last_hidden = x[:, -1, :]
        
        rewards = self.reward_head(last_hidden).squeeze(-1)
        return rewards


def compute_preference_loss(reward_chosen, reward_rejected):
    """Bradley-Terry preference loss.
    
    P(chosen > rejected) = sigmoid(r_chosen - r_rejected)
    Loss = -log P(chosen > rejected)
         = -log sigmoid(r_chosen - r_rejected)
    
    Args:
        reward_chosen: [batch] scalar rewards for chosen responses
        reward_rejected: [batch] scalar rewards for rejected responses
    Returns:
        loss: scalar, mean preference loss
        accuracy: fraction where reward_chosen > reward_rejected
    """
    loss = -F.logsigmoid(reward_chosen - reward_rejected).mean()
    
    with torch.no_grad():
        accuracy = (reward_chosen > reward_rejected).float().mean()
    
    return loss, accuracy


# Test it
rm = RewardModel(vocab_size=1000, embed_dim=128)
dummy_ids = torch.randint(0, 1000, (4, 32))
rewards = rm(dummy_ids)
print(f"Reward model output shape: {rewards.shape}")
print(f"Sample rewards: {rewards.detach().numpy()}")
print(f"\nTotal parameters: {sum(p.numel() for p in rm.parameters()):,}")

In [ ]:
class PreferenceDataset(Dataset):
    """Synthetic preference dataset for reward model training.
    
    Generates (chosen, rejected) pairs where chosen sequences have
    a pattern that correlates with higher reward.
    """
    
    def __init__(self, n_samples=500, seq_len=32, vocab_size=1000):
        self.data = []
        for _ in range(n_samples):
            prompt = torch.randint(0, vocab_size, (seq_len // 2,))
            # Chosen: lower token IDs (proxy for 'better quality')
            chosen_resp = torch.randint(0, vocab_size // 2, (seq_len // 2,))
            # Rejected: higher token IDs (proxy for 'worse quality')
            rejected_resp = torch.randint(vocab_size // 2, vocab_size, (seq_len // 2,))
            
            chosen = torch.cat([prompt, chosen_resp])
            rejected = torch.cat([prompt, rejected_resp])
            self.data.append((chosen, rejected))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def train_reward_model(rm, dataset, n_epochs=10, lr=1e-3, batch_size=32):
    """Train the reward model on preference pairs.
    
    Returns training history for visualization.
    """
    optimizer = optim.Adam(rm.parameters(), lr=lr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    history = {"loss": [], "accuracy": []}
    
    for epoch in range(n_epochs):
        epoch_losses, epoch_accs = [], []
        
        for chosen, rejected in loader:
            chosen, rejected = chosen.to(device), rejected.to(device)
            
            r_chosen = rm(chosen)
            r_rejected = rm(rejected)
            
            loss, acc = compute_preference_loss(r_chosen, r_rejected)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_losses.append(loss.item())
            epoch_accs.append(acc.item())
        
        avg_loss = np.mean(epoch_losses)
        avg_acc = np.mean(epoch_accs)
        history["loss"].append(avg_loss)
        history["accuracy"].append(avg_acc)
        
        if epoch % 2 == 0:
            print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Accuracy={avg_acc:.2%}")
    
    return history


# Train the reward model
print("Training Reward Model...")
print("=" * 50)
rm = RewardModel(vocab_size=1000, embed_dim=128).to(device)
pref_dataset = PreferenceDataset(n_samples=500)
rm_history = train_reward_model(rm, pref_dataset, n_epochs=10)
print("\nReward model trained!")

In [ ]:
# Visualize reward model training
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curve
axes[0].plot(rm_history["loss"], 'b-', linewidth=2)
axes[0].set_title("Reward Model Training Loss", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Bradley-Terry Loss")
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(rm_history["accuracy"], 'g-', linewidth=2)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
axes[1].set_title("Preference Prediction Accuracy", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Reward distribution
rm_for_viz = rm
rm_for_viz.train(False)
with torch.no_grad():
    chosen_rewards, rejected_rewards = [], []
    for chosen, rejected in DataLoader(pref_dataset, batch_size=64):
        chosen_rewards.extend(rm_for_viz(chosen.to(device)).cpu().numpy())
        rejected_rewards.extend(rm_for_viz(rejected.to(device)).cpu().numpy())

axes[2].hist(chosen_rewards, bins=30, alpha=0.6, label='Chosen', color='green')
axes[2].hist(rejected_rewards, bins=30, alpha=0.6, label='Rejected', color='red')
axes[2].set_title("Reward Distribution (After Training)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Reward Score")
axes[2].set_ylabel("Count")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMean chosen reward:   {np.mean(chosen_rewards):.4f}")
print(f"Mean rejected reward: {np.mean(rejected_rewards):.4f}")
print(f"Separation: {np.mean(chosen_rewards) - np.mean(rejected_rewards):.4f}")

---

## 3. Part 2: GAE (Generalized Advantage Estimation)

GAE is the secret weapon of PPO. It smoothly trades off between bias and variance in advantage estimation.

### The Problem

We need to compute **advantages** A(s, a) = "how much better was this action than expected?"

Two extremes:
- **TD(0)** (lambda=0): A = r + gamma*V(s') - V(s). **Low variance, high bias** (relies on V being accurate).
- **Monte Carlo** (lambda=1): A = R - V(s), where R is total return. **High variance, zero bias** (uses actual rewards).

### The GAE Formula

GAE_t = sum over l from 0 to T-t of (gamma * lambda)^l * delta_{t+l}

where delta_t = r_t + gamma * V(s_{t+1}) - V(s_t) is the TD error.

- **lambda = 0**: Pure TD (biased but stable)
- **lambda = 1**: Pure MC (unbiased but noisy)
- **lambda = 0.95**: Sweet spot (standard default)

### FAANG Interview Question

**Q: "Explain GAE and the role of lambda. When would you change it?"**

**A**: GAE computes advantages as an exponentially-weighted average of n-step TD errors. Lambda controls the bias-variance tradeoff: lower lambda = more bias but less variance (good when your value function is accurate), higher lambda = less bias but more variance (good when your value function is poor, e.g., early in training). In practice, lambda=0.95 works for most tasks. You might lower lambda if training is noisy (high variance), or raise it if the value function is inaccurate.

In [ ]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Compute Generalized Advantage Estimation.
    
    Walks backward through the trajectory, computing the exponentially-weighted
    sum of TD errors. This gives us advantages that smoothly interpolate between
    TD(0) (lambda=0) and Monte Carlo (lambda=1).
    
    Args:
        rewards: [T] rewards at each timestep
        values: [T+1] value estimates (includes bootstrap value at T+1)
        dones: [T] whether episode ended at each timestep
        gamma: discount factor (default 0.99)
        lam: GAE lambda for bias-variance tradeoff (default 0.95)
    Returns:
        advantages: [T] GAE advantages
        returns: [T] advantage + value = target for value function
    """
    T = len(rewards)
    advantages = torch.zeros(T)
    last_gae = 0.0
    
    # Walk backward through trajectory
    for t in reversed(range(T)):
        # TD error: r + gamma * V(s') - V(s)
        # If done, next value is 0 (episode ended)
        next_value = values[t + 1] * (1.0 - dones[t])
        delta = rewards[t] + gamma * next_value - values[t]
        
        # GAE: accumulate discounted TD errors
        last_gae = delta + gamma * lam * (1.0 - dones[t]) * last_gae
        advantages[t] = last_gae
    
    # Returns = advantages + values (target for critic)
    returns = advantages + values[:-1]
    
    return advantages, returns


# Test with a simple trajectory
test_rewards = torch.tensor([1.0, 0.0, 0.0, 0.0, 10.0])  # sparse reward at end
test_values = torch.tensor([0.5, 0.3, 0.2, 0.1, 0.8, 0.0])  # T+1 values
test_dones = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])

advantages, returns = compute_gae(test_rewards, test_values, test_dones)
print("Test Trajectory:")
print(f"  Rewards:    {test_rewards.numpy()}")
print(f"  Values:     {test_values[:-1].numpy()}")
print(f"  Advantages: {advantages.numpy().round(4)}")
print(f"  Returns:    {returns.numpy().round(4)}")

In [ ]:
# Visualize GAE at different lambda values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Generate a longer trajectory for visualization
T = 50
np.random.seed(42)
viz_rewards = torch.tensor(np.random.randn(T) * 0.1)
viz_rewards[-1] = 5.0  # big reward at end
viz_values = torch.tensor(np.linspace(0.1, 1.0, T + 1).astype(np.float32))
viz_dones = torch.zeros(T)
viz_dones[-1] = 1.0

lambdas = [0.0, 0.5, 0.9, 0.95, 1.0]
colors = ['red', 'orange', 'blue', 'green', 'purple']

for lam_val, color in zip(lambdas, colors):
    adv, _ = compute_gae(viz_rewards, viz_values, viz_dones, gamma=0.99, lam=lam_val)
    axes[0].plot(adv.numpy(), color=color, alpha=0.8, linewidth=1.5, label=f'lambda={lam_val}')

axes[0].set_title('GAE Advantages at Different Lambda Values', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Advantage')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Variance of advantages for each lambda
lambda_range = np.linspace(0, 1, 20)
variances = []
for lam_val in lambda_range:
    adv, _ = compute_gae(viz_rewards, viz_values, viz_dones, gamma=0.99, lam=lam_val)
    variances.append(adv.var().item())

axes[1].plot(lambda_range, variances, 'b-', linewidth=2)
axes[1].axvline(x=0.95, color='green', linestyle='--', alpha=0.7, label='Default lambda=0.95')
axes[1].set_title('Advantage Variance vs Lambda', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lambda')
axes[1].set_ylabel('Variance of Advantages')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Higher lambda = higher variance but less bias.")
print("lambda=0.95 is the standard default - a good balance.")

---

## 4. Part 3: PPO for Language Models

Now the main event. PPO for LLMs is fundamentally different from PPO for CartPole:

| CartPole PPO | LLM PPO |
|:---|:---|
| State = 4 floats | State = sequence of tokens |
| Action = 0 or 1 | Action = next token (50K+ vocab) |
| Reward at every step | Reward only at end of response |
| Episode = ~200 steps | Episode = ~100-500 tokens |
| No reference model | Reference model + KL penalty |
| 1 model | 4 models (policy, ref, reward, value) |

### The 4-Model Dance

At each PPO step:
1. **Policy Model** generates a response -> collect log-probs
2. **Reference Model** (frozen) computes log-probs of the same response -> for KL penalty
3. **Reward Model** (frozen) scores the response -> the signal we optimize
4. **Value Head** (on policy) estimates V(s) at each token position -> for GAE

### FAANG Interview Question

**Q: "Why do we need a reference model in RLHF?"**

**A**: Without the reference model, the policy would overfit to the reward model — finding adversarial responses that score high but are low quality (reward hacking). The KL penalty KL(pi || pi_ref) constrains the policy to stay close to the SFT model, which already produces reasonable text. This acts as a regularizer: the policy can improve preferences but can't deviate too far into gibberish. Mathematically, the effective reward becomes r(x, y) - beta * KL(pi || pi_ref), where beta controls the constraint strength.

In [ ]:
class SmallLM(nn.Module):
    """Small language model for demonstrating PPO training.
    
    A miniature GPT-style model with both a policy (LM) head and a value head.
    In production, this would be GPT-2/LLaMA with the value head added on top.
    
    Architecture:
        Token IDs -> Embedding + PosEmbed -> TransformerEncoder -> {
            LM Head:    -> Linear(hidden, vocab) -> logits
            Value Head: -> Linear(hidden, 1) -> values
        }
    """
    
    def __init__(self, vocab_size=1000, embed_dim=128, n_heads=4,
                 n_layers=2, max_seq_len=64):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        
        # Policy head (LM head) - generates tokens
        self.lm_head = nn.Linear(embed_dim, vocab_size)
        
        # Value head - estimates V(s) at each token position
        self.value_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )
    
    def forward(self, input_ids):
        """Forward pass returning both logits and values.
        
        Args:
            input_ids: [batch, seq_len] token indices
        Returns:
            logits: [batch, seq_len, vocab_size] for next token prediction
            values: [batch, seq_len] state value at each position
        """
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        
        # Causal mask to prevent attending to future tokens
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_ids.device), diagonal=1
        ).bool()
        
        x = self.transformer(x, mask=causal_mask)
        
        logits = self.lm_head(x)
        values = self.value_head(x).squeeze(-1)
        
        return logits, values
    
    def generate(self, prompt_ids, max_new_tokens=16, temperature=1.0):
        """Autoregressive generation with log-prob collection.
        
        Args:
            prompt_ids: [batch, prompt_len] prompt token indices
            max_new_tokens: how many tokens to generate
            temperature: sampling temperature (1.0 = default)
        Returns:
            sequences: [batch, prompt_len + max_new_tokens] full sequences
            log_probs: [batch, max_new_tokens] log prob of each generated token
            values: [batch, max_new_tokens] value estimates at each step
        """
        was_training = self.training
        self.train(False)
        sequences = prompt_ids.clone()
        all_log_probs = []
        all_values = []
        
        with torch.no_grad():
            for _ in range(max_new_tokens):
                logits, values = self.forward(sequences)
                
                # Get logits for the last position
                next_logits = logits[:, -1, :] / temperature
                next_value = values[:, -1]
                
                # Sample next token
                probs = F.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                
                # Compute log prob of sampled token
                log_prob = F.log_softmax(next_logits, dim=-1)
                selected_log_prob = log_prob.gather(1, next_token).squeeze(-1)
                
                all_log_probs.append(selected_log_prob)
                all_values.append(next_value)
                
                sequences = torch.cat([sequences, next_token], dim=1)
        
        if was_training:
            self.train(True)
        return (
            sequences,
            torch.stack(all_log_probs, dim=1),
            torch.stack(all_values, dim=1),
        )


# Test the model
lm = SmallLM(vocab_size=1000, embed_dim=128).to(device)
prompt = torch.randint(0, 1000, (2, 8)).to(device)
seqs, log_probs, values = lm.generate(prompt, max_new_tokens=16)

print(f"Prompt shape:    {prompt.shape}")
print(f"Generated shape: {seqs.shape}")
print(f"Log probs shape: {log_probs.shape}")
print(f"Values shape:    {values.shape}")
print(f"\nSample log probs: {log_probs[0].cpu().numpy().round(3)}")

In [ ]:
@dataclass
class ExperienceBuffer:
    """Stores rollout experience for PPO mini-batch training.
    
    After generating responses and scoring with reward model,
    we store everything needed for PPO updates in this buffer.
    Then we iterate over mini-batches for multiple PPO epochs.
    """
    queries: List[torch.Tensor] = field(default_factory=list)
    responses: List[torch.Tensor] = field(default_factory=list)
    old_log_probs: List[torch.Tensor] = field(default_factory=list)
    rewards: List[torch.Tensor] = field(default_factory=list)
    values: List[torch.Tensor] = field(default_factory=list)
    advantages: List[torch.Tensor] = field(default_factory=list)
    returns: List[torch.Tensor] = field(default_factory=list)
    
    def add(self, query, response, log_probs, reward, values):
        """Add a single rollout to the buffer."""
        self.queries.append(query)
        self.responses.append(response)
        self.old_log_probs.append(log_probs)
        self.rewards.append(reward)
        self.values.append(values)
    
    def compute_advantages(self, gamma=0.99, lam=0.95):
        """Compute GAE advantages for all stored experiences."""
        self.advantages = []
        self.returns = []
        
        for reward_seq, value_seq in zip(self.rewards, self.values):
            # For LLM PPO: reward is only at last token
            T = len(value_seq)
            per_token_rewards = torch.zeros(T)
            per_token_rewards[-1] = reward_seq  # reward at final token only
            
            # Add bootstrap value of 0 (episode ends after response)
            values_with_bootstrap = torch.cat([value_seq, torch.tensor([0.0])])
            dones = torch.zeros(T)
            dones[-1] = 1.0
            
            adv, ret = compute_gae(per_token_rewards, values_with_bootstrap, dones, gamma, lam)
            self.advantages.append(adv)
            self.returns.append(ret)
    
    def get_minibatches(self, batch_size=4):
        """Yield mini-batches for PPO training."""
        n = len(self.queries)
        indices = np.random.permutation(n)
        
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]
            yield {
                "queries": [self.queries[i] for i in batch_idx],
                "responses": [self.responses[i] for i in batch_idx],
                "old_log_probs": [self.old_log_probs[i] for i in batch_idx],
                "advantages": [self.advantages[i] for i in batch_idx],
                "returns": [self.returns[i] for i in batch_idx],
            }
    
    def clear(self):
        """Clear the buffer for next rollout phase."""
        for attr in ['queries', 'responses', 'old_log_probs', 
                      'rewards', 'values', 'advantages', 'returns']:
            getattr(self, attr).clear()
    
    def __len__(self):
        return len(self.queries)


print("ExperienceBuffer defined!")
print("This stores rollout data for mini-batch PPO updates.")

In [ ]:
class PPOTrainer:
    """PPO Trainer for language model alignment.
    
    Implements the full PPO loop:
    1. Generate responses with policy
    2. Score with reward model
    3. Compute KL penalty against reference
    4. Compute advantages with GAE
    5. PPO update (clipped objective + value loss + entropy bonus)
    """
    
    def __init__(self, policy_model, ref_model, reward_model,
                 lr=1e-5, kl_coeff=0.1, clip_epsilon=0.2,
                 value_coeff=0.5, entropy_coeff=0.01,
                 max_grad_norm=1.0, ppo_epochs=4):
        self.policy = policy_model
        self.ref = ref_model
        self.reward_model = reward_model
        
        # Freeze reference and reward models
        for param in self.ref.parameters():
            param.requires_grad = False
        for param in self.reward_model.parameters():
            param.requires_grad = False
        
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.kl_coeff = kl_coeff
        self.clip_epsilon = clip_epsilon
        self.value_coeff = value_coeff
        self.entropy_coeff = entropy_coeff
        self.max_grad_norm = max_grad_norm
        self.ppo_epochs = ppo_epochs
        self.buffer = ExperienceBuffer()
    
    def compute_kl_penalty(self, policy_logits, ref_logits):
        """Compute per-token KL divergence between policy and reference."""
        policy_probs = F.softmax(policy_logits, dim=-1)
        policy_log_probs = F.log_softmax(policy_logits, dim=-1)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        
        kl = (policy_probs * (policy_log_probs - ref_log_probs)).sum(dim=-1)
        return kl
    
    def rollout(self, prompts, max_new_tokens=16):
        """Generate responses and collect experience."""
        self.policy.train(False)
        
        with torch.no_grad():
            # Generate with policy
            sequences, old_log_probs, values = self.policy.generate(
                prompts, max_new_tokens=max_new_tokens
            )
            
            # Score with reward model
            rewards = self.reward_model(sequences)
            
            # Compute KL penalty
            policy_logits, _ = self.policy(sequences)
            ref_logits, _ = self.ref(sequences)
            kl = self.compute_kl_penalty(policy_logits, ref_logits)
            
            # KL penalty on the response tokens only
            prompt_len = prompts.shape[1]
            kl_penalty = kl[:, prompt_len:].mean(dim=1)
            
            # Adjusted reward = reward - kl_coeff * KL
            adjusted_rewards = rewards - self.kl_coeff * kl_penalty
        
        # Store in buffer (per-example)
        for i in range(prompts.shape[0]):
            self.buffer.add(
                query=prompts[i],
                response=sequences[i, prompt_len:],
                log_probs=old_log_probs[i],
                reward=adjusted_rewards[i],
                values=values[i]
            )
        
        self.policy.train(True)
        return {
            "mean_reward": rewards.mean().item(),
            "mean_kl": kl_penalty.mean().item(),
            "mean_adjusted_reward": adjusted_rewards.mean().item()
        }
    
    def ppo_step(self, batch):
        """Single PPO optimization step on a mini-batch."""
        total_policy_loss = 0.0
        total_value_loss = 0.0
        total_entropy = 0.0
        n = len(batch["queries"])
        
        for i in range(n):
            query = batch["queries"][i].unsqueeze(0).to(device)
            response = batch["responses"][i].unsqueeze(0).to(device)
            old_lp = batch["old_log_probs"][i].to(device)
            advantages = batch["advantages"][i].to(device)
            ret = batch["returns"][i].to(device)
            
            # Normalize advantages (PPO best practice)
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            
            # Forward pass through policy
            full_seq = torch.cat([query, response], dim=1)
            logits, values = self.policy(full_seq)
            
            # Get log probs for response tokens
            prompt_len = query.shape[1]
            resp_logits = logits[:, prompt_len-1:-1, :]
            resp_log_probs = F.log_softmax(resp_logits, dim=-1)
            new_lp = resp_log_probs.gather(
                2, response.unsqueeze(-1)
            ).squeeze(-1).squeeze(0)
            
            # PPO clipped objective
            ratio = torch.exp(new_lp - old_lp)
            obj_unclipped = ratio * advantages
            obj_clipped = torch.clamp(
                ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon
            ) * advantages
            policy_loss = -torch.min(obj_unclipped, obj_clipped).mean()
            
            # Value loss
            resp_values = values[:, prompt_len:].squeeze(0)
            min_len = min(len(resp_values), len(ret))
            value_loss = F.mse_loss(resp_values[:min_len], ret[:min_len])
            
            # Entropy bonus (encourages exploration)
            probs = F.softmax(resp_logits, dim=-1)
            entropy = -(probs * resp_log_probs).sum(dim=-1).mean()
            
            total_policy_loss += policy_loss
            total_value_loss += value_loss
            total_entropy += entropy
        
        # Combined loss
        loss = (
            total_policy_loss / n
            + self.value_coeff * total_value_loss / n
            - self.entropy_coeff * total_entropy / n
        )
        
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
        self.optimizer.step()
        
        return {
            "policy_loss": (total_policy_loss / n).item(),
            "value_loss": (total_value_loss / n).item(),
            "entropy": (total_entropy / n).item(),
            "total_loss": loss.item()
        }
    
    def train_step(self, prompts, max_new_tokens=16):
        """Full PPO training step: rollout + optimize."""
        # Phase 1: Rollout
        self.buffer.clear()
        rollout_metrics = self.rollout(prompts, max_new_tokens)
        
        # Phase 2: Compute advantages
        self.buffer.compute_advantages(gamma=0.99, lam=0.95)
        
        # Phase 3: PPO optimization
        all_losses = []
        for epoch in range(self.ppo_epochs):
            for batch in self.buffer.get_minibatches(batch_size=4):
                losses = self.ppo_step(batch)
                all_losses.append(losses)
        
        avg_losses = {
            k: np.mean([l[k] for l in all_losses]) for k in all_losses[0]
        }
        
        return {**rollout_metrics, **avg_losses}


print("PPOTrainer defined!")
print("This orchestrates the full RLHF training loop.")

In [ ]:
# ============================================================
# Full PPO Training Loop for Language Models
# ============================================================

print("Starting PPO Training on a Small Language Model")
print("=" * 55)

# Initialize models
VOCAB_SIZE = 1000
EMBED_DIM = 128

policy_model = SmallLM(VOCAB_SIZE, EMBED_DIM).to(device)
ref_model = copy.deepcopy(policy_model).to(device)
reward_model_for_ppo = rm  # Use the trained reward model from Part 1

# Initialize trainer
trainer = PPOTrainer(
    policy_model=policy_model,
    ref_model=ref_model,
    reward_model=reward_model_for_ppo,
    lr=1e-4,
    kl_coeff=0.1,
    clip_epsilon=0.2,
    ppo_epochs=2
)

# Training loop
n_steps = 8
batch_size = 8
prompt_len = 8
max_new_tokens = 16

training_history = []

for step in range(n_steps):
    # Generate random prompts (in production, from a dataset)
    prompts = torch.randint(0, VOCAB_SIZE, (batch_size, prompt_len)).to(device)
    
    # Run one PPO step
    metrics = trainer.train_step(prompts, max_new_tokens=max_new_tokens)
    training_history.append(metrics)
    
    print(f"Step {step+1}/{n_steps}: "
          f"Reward={metrics['mean_reward']:.4f}, "
          f"KL={metrics['mean_kl']:.4f}, "
          f"Policy Loss={metrics['policy_loss']:.4f}")

print("\nPPO training complete!")

In [ ]:
# ============================================================
# Publication-Quality 4-Panel Visualization
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
steps = range(1, len(training_history) + 1)

# Panel 1: Mean Reward Over Training
rewards_hist = [h['mean_reward'] for h in training_history]
adjusted_hist = [h['mean_adjusted_reward'] for h in training_history]
axes[0, 0].plot(steps, rewards_hist, 'g-o', linewidth=2, label='Raw Reward')
axes[0, 0].plot(steps, adjusted_hist, 'b--s', linewidth=2, label='Adjusted (- KL)')
axes[0, 0].set_title('Reward Over Training', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('PPO Step')
axes[0, 0].set_ylabel('Mean Reward')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Panel 2: KL Divergence
kls_hist = [h['mean_kl'] for h in training_history]
axes[0, 1].plot(steps, kls_hist, 'r-o', linewidth=2)
axes[0, 1].axhline(y=0, color='gray', linestyle=':', alpha=0.5)
axes[0, 1].set_title('KL Divergence (Policy vs Reference)', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('PPO Step')
axes[0, 1].set_ylabel('Mean KL')
axes[0, 1].grid(True, alpha=0.3)

# Panel 3: Loss Components
p_loss_hist = [h['policy_loss'] for h in training_history]
v_loss_hist = [h['value_loss'] for h in training_history]
axes[1, 0].plot(steps, p_loss_hist, 'b-o', linewidth=2, label='Policy Loss')
axes[1, 0].plot(steps, v_loss_hist, 'orange', marker='s', linewidth=2, label='Value Loss')
axes[1, 0].set_title('PPO Loss Components', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('PPO Step')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Panel 4: Entropy
entropy_hist = [h['entropy'] for h in training_history]
axes[1, 1].plot(steps, entropy_hist, 'purple', marker='D', linewidth=2)
axes[1, 1].set_title('Policy Entropy (Exploration)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('PPO Step')
axes[1, 1].set_ylabel('Entropy')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('RLHF with PPO Training Dynamics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nWhat to look for:")
print("  - Reward should increase (model is improving)")
print("  - KL should stay moderate (not diverging from reference)")
print("  - Policy loss should decrease then stabilize")
print("  - Entropy should gently decrease (model becomes more confident)")

---

## 5. Part 4: End-to-End RLHF Pipeline

Let's now put all three stages together into one clean pipeline class.

### FAANG Interview Question

**Q: "Walk through a complete RLHF pipeline. What are the key decisions at each stage?"**

**A**:

**Stage 1 - SFT**: Fine-tune the base model on high-quality (instruction, response) pairs. Key decisions: data quality > quantity; 1-3 epochs to avoid overfitting; learning rate ~1e-5.

**Stage 2 - Reward Model**: Train on preference pairs (chosen > rejected). Key decisions: same architecture as policy (share the backbone); Bradley-Terry loss; 50K-100K preference pairs; careful annotation guidelines.

**Stage 3 - PPO**: Generate -> Score -> Optimize. Key decisions: KL coefficient (start high ~0.2, anneal to ~0.05); mini-batch size; number of PPO epochs (2-4); gradient clipping; learning rate ~1e-6 (very small).

**Monitoring**: Track reward (should increase), KL (should stay bounded), response quality (human assessment), and check for reward hacking (reward increases but quality decreases).

In [ ]:
class RLHFPipeline:
    """End-to-end RLHF Pipeline: SFT -> Reward Model -> PPO.
    
    Orchestrates the full alignment process.
    In production, each stage would be a separate job on a GPU cluster.
    Here we run everything sequentially on a single device.
    """
    
    def __init__(self, vocab_size=1000, embed_dim=128):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.history = {"sft": None, "rm": None, "ppo": None}
    
    def stage1_sft(self, n_epochs=5):
        """Stage 1: Supervised Fine-Tuning."""
        print("\n" + "=" * 55)
        print("Stage 1: Supervised Fine-Tuning (SFT)")
        print("=" * 55)
        
        self.sft_model = SmallLM(self.vocab_size, self.embed_dim).to(device)
        optimizer = optim.Adam(self.sft_model.parameters(), lr=1e-3)
        sft_losses = []
        
        for epoch in range(n_epochs):
            data = torch.randint(0, self.vocab_size, (32, 32)).to(device)
            logits, _ = self.sft_model(data[:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, self.vocab_size), data[:, 1:].reshape(-1)
            )
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            sft_losses.append(loss.item())
            
            if epoch % 2 == 0:
                print(f"  SFT Epoch {epoch}: Loss={loss.item():.4f}")
        
        self.history["sft"] = sft_losses
        print("  SFT complete!")
        return self.sft_model
    
    def stage2_reward_model(self, n_epochs=8):
        """Stage 2: Train Reward Model on preference data."""
        print("\n" + "=" * 55)
        print("Stage 2: Reward Model Training")
        print("=" * 55)
        
        self.rm = RewardModel(
            self.vocab_size, self.embed_dim
        ).to(device)
        dataset = PreferenceDataset(n_samples=300)
        self.history["rm"] = train_reward_model(
            self.rm, dataset, n_epochs=n_epochs
        )
        print("  Reward model trained!")
        return self.rm
    
    def stage3_ppo(self, n_steps=6):
        """Stage 3: PPO training using reward model."""
        print("\n" + "=" * 55)
        print("Stage 3: PPO Alignment")
        print("=" * 55)
        
        policy = copy.deepcopy(self.sft_model).to(device)
        ref = copy.deepcopy(self.sft_model).to(device)
        
        ppo_trainer = PPOTrainer(
            policy_model=policy,
            ref_model=ref,
            reward_model=self.rm,
            lr=1e-4,
            kl_coeff=0.1,
            ppo_epochs=2
        )
        
        ppo_history = []
        for step in range(n_steps):
            prompts = torch.randint(
                0, self.vocab_size, (8, 8)
            ).to(device)
            metrics = ppo_trainer.train_step(prompts, max_new_tokens=16)
            ppo_history.append(metrics)
            print(f"  PPO Step {step+1}: "
                  f"Reward={metrics['mean_reward']:.4f}, "
                  f"KL={metrics['mean_kl']:.4f}")
        
        self.history["ppo"] = ppo_history
        self.aligned_model = policy
        print("  PPO alignment complete!")
        return policy
    
    def run_full_pipeline(self):
        """Run the complete RLHF pipeline."""
        print("\nRunning Full RLHF Pipeline")
        self.stage1_sft()
        self.stage2_reward_model()
        self.stage3_ppo()
        print("\n" + "=" * 55)
        print("RLHF Pipeline Complete!")
        print("=" * 55)
        return self.history


# Run the full pipeline
pipeline = RLHFPipeline(vocab_size=1000, embed_dim=128)
full_history = pipeline.run_full_pipeline()

In [ ]:
# ============================================================
# Pipeline Summary Visualization
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stage 1: SFT Loss
axes[0].plot(full_history["sft"], 'b-o', linewidth=2, markersize=6)
axes[0].set_title('Stage 1: SFT Training', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Next-token prediction', xy=(0.5, 0.95), xycoords='axes fraction',
                 ha='center', fontsize=10, style='italic', color='gray')

# Stage 2: Reward Model
axes[1].plot(full_history["rm"]["loss"], 'r-o', linewidth=2, label='Loss')
ax1_twin = axes[1].twinx()
ax1_twin.plot(full_history["rm"]["accuracy"], 'g-s', linewidth=2, label='Accuracy')
ax1_twin.set_ylabel('Accuracy', color='green')
axes[1].set_title('Stage 2: Reward Model', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Bradley-Terry Loss', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].annotate('Preference learning', xy=(0.5, 0.95), xycoords='axes fraction',
                 ha='center', fontsize=10, style='italic', color='gray')

# Stage 3: PPO
ppo_rewards_viz = [h['mean_reward'] for h in full_history["ppo"]]
ppo_kl_viz = [h['mean_kl'] for h in full_history["ppo"]]
axes[2].plot(ppo_rewards_viz, 'g-o', linewidth=2, label='Reward')
ax2_twin = axes[2].twinx()
ax2_twin.plot(ppo_kl_viz, 'r--s', linewidth=2, label='KL')
ax2_twin.set_ylabel('KL Divergence', color='red')
axes[2].set_title('Stage 3: PPO Alignment', fontsize=13, fontweight='bold')
axes[2].set_xlabel('PPO Step')
axes[2].set_ylabel('Mean Reward', color='green')
axes[2].grid(True, alpha=0.3)
axes[2].annotate('RL fine-tuning', xy=(0.5, 0.95), xycoords='axes fraction',
                 ha='center', fontsize=10, style='italic', color='gray')

plt.suptitle('Complete RLHF Pipeline: SFT -> Reward Model -> PPO', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Practical Considerations for Production RLHF

### KL Coefficient Tuning
- **Too high** (beta > 0.5): Policy barely changes. Alignment is weak.
- **Too low** (beta < 0.01): Policy diverges. Reward hacking occurs.
- **Sweet spot**: Start at beta=0.2, anneal to beta=0.05 over training.
- **Adaptive KL**: Track KL and adjust beta to keep KL near a target (e.g., 6.0). OpenAI does this.

### Reference Model Management
- **Deep copy** (standard): Freeze a copy of the SFT model. Simple but doubles memory.
- **EMA (Exponential Moving Average)**: ref = alpha * ref + (1-alpha) * policy. Smooth updates. Used by some teams.
- **Periodic refresh**: Update the reference model every N steps to the current policy. More aggressive.

### Reward Normalization
- Reward model outputs can drift in scale. Normalize rewards to zero mean, unit variance per batch.
- This prevents the value function from becoming miscalibrated.

### Scaling to 70B+ Models
- **Separate machines**: Policy on GPU cluster A, Reward Model on GPU cluster B. Communicate via gRPC.
- **Tensor parallelism**: Split model across GPUs within a machine.
- **vLLM for generation**: Use optimized inference engines for the rollout phase (generation is the bottleneck).
- **Mixed precision**: Generate in fp16, train in bf16 with fp32 master weights.
- **Gradient accumulation**: Effective batch sizes of 256-1024 for stable PPO.

---

## 7. FAANG Interview Questions

### Q1: "How do you detect reward hacking? Give concrete examples."

**A**: Reward hacking is when the model exploits the reward model rather than genuinely improving.

**Detection**:
- Monitor reward vs human preference quality. If reward increases but human preference scores don't, that's hacking.
- Track output diversity (unique n-grams). Dropping diversity = mode collapse.
- Check response length distribution. A sudden shift toward very long/short responses is suspicious.

**Examples**:
- The model repeats phrases the reward model likes ("I'd be happy to help!" 50 times)
- The model generates very long responses (if the RM was biased toward length)
- The model produces confident-sounding but factually wrong answers

**Prevention**: KL penalty, reward model ensembles, periodic human assessment.

---

### Q2: "Compare PPO-Clip vs PPO-Penalty. Which is better?"

**A**:
- **PPO-Clip**: Clips the ratio r(theta) to [1-epsilon, 1+epsilon]. Hard constraint. Simple.
- **PPO-Penalty**: Adds KL divergence as a penalty term. Soft constraint. Adaptive.

In practice, **PPO-Clip is standard** because:
1. No extra hyperparameter for the penalty coefficient
2. More robust to different reward scales
3. The clipping is per-token, giving fine-grained control

Note: The KL penalty against the reference model (for RLHF) is separate from PPO-Penalty. Even PPO-Clip uses a KL penalty against the reference.

---

### Q3: "Why normalize advantages? What happens if you don't?"

**A**: Without normalization, advantages can have large magnitude and variance across batches. This causes:
1. **Unstable gradients**: Large advantages lead to large policy updates which cause training divergence
2. **Ratio explosion**: When advantages are very large, the policy ratio can overflow
3. **Bias toward recent batches**: If one batch has higher rewards, it dominates the update

Normalization: A = (A - mean(A)) / (std(A) + epsilon). This ensures roughly half the advantages are positive (good actions) and half negative (bad actions), giving balanced gradient signals.

---

### Q4: "How do you scale RLHF to a 70B parameter model?"

**A**: The key bottleneck is that RLHF requires 4 models in memory simultaneously.

**Solutions**:
1. **Separate serving**: Policy generation on one cluster, reward scoring on another. Use async communication.
2. **Parameter offloading**: Keep reference model on CPU, move to GPU only for KL computation.
3. **DeepSpeed ZeRO-3**: Shard model parameters across GPUs.
4. **LoRA for policy**: Only train low-rank adapters. Reference model = base model (no extra copy).
5. **vLLM/TGI for generation**: The rollout phase (generating responses) is the bottleneck. Use optimized inference.

**Typical setup**: 64-256 GPUs. Policy + Value on 16-32 GPUs (FSDP). Reward Model on 8-16 GPUs (inference only). Reference = same weights as policy minus LoRA adapters.

---

### Q5: "What are the main failure modes of RLHF?"

**A**:
1. **Reward hacking**: Model exploits reward model weaknesses
2. **Mode collapse**: Model converges to generating the same response for all prompts
3. **Sycophancy**: Model agrees with the user regardless of correctness
4. **Reward model miscalibration**: RM scores don't correlate with actual human preferences
5. **KL divergence explosion**: Policy drifts too far from reference
6. **Training instability**: PPO is inherently unstable; small hyperparameter changes cause large behavioral shifts

In [ ]:
# ============================================================
# Interview Coding Challenge: Implement a simplified PPO loss
# ============================================================

# Challenge: Implement the PPO clipped objective from memory.
# No peeking at the code above!

def ppo_clipped_loss(old_log_probs, new_log_probs, advantages, epsilon=0.2):
    """PPO Clipped Objective - the core of modern RL.
    
    Steps:
    1. Compute probability ratio: r = exp(new_lp - old_lp)
    2. Unclipped objective: r * A
    3. Clipped objective: clip(r, 1-eps, 1+eps) * A
    4. Loss = -min(unclipped, clipped).mean()
    """
    ratio = torch.exp(new_log_probs - old_log_probs)
    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * advantages
    return -torch.min(unclipped, clipped).mean()


# Verify
old_lp = torch.tensor([-1.5, -2.0, -1.0])
new_lp = torch.tensor([-1.2, -2.5, -0.8])
adv = torch.tensor([1.0, -0.5, 2.0])

loss = ppo_clipped_loss(old_lp, new_lp, adv)
print(f"PPO Loss: {loss.item():.4f}")
print("\nIf you can write this from memory on a whiteboard, you're ready.")

---

## 8. Key Takeaways

### Foundation (Know These Cold)
- [ ] The 3-stage pipeline: Pre-training -> SFT -> Alignment (RLHF or DPO)
- [ ] Reward Model: Bradley-Terry loss, last-token pooling, preference pair format
- [ ] GAE: lambda controls bias-variance, default lambda=0.95, computed backward through trajectory
- [ ] PPO for LLMs requires 4 models: Policy, Reference, Reward, Value head

### Implementation (Be Able to Code)
- [ ] `compute_preference_loss()` - Bradley-Terry: -log sigmoid(r_chosen - r_rejected)
- [ ] `compute_gae()` - backward walk, accumulate (gamma * lambda)^l * delta_t
- [ ] `ppo_clipped_loss()` - ratio, clip, min, negate
- [ ] `compute_kl_penalty()` - KL(policy || reference) per token

### Production Patterns (Interview Differentiator)
- [ ] KL coefficient annealing (start high, reduce)
- [ ] Reward normalization (zero mean, unit variance per batch)
- [ ] Reference model strategies (deep copy, EMA, periodic refresh)
- [ ] Reward hacking detection (reward vs human assessment divergence)
- [ ] Scaling: LoRA, DeepSpeed, vLLM, separate serving clusters

### Common Mistakes (Avoid These)
- [ ] Forgetting to freeze the reference model
- [ ] Not normalizing advantages
- [ ] Using too low a KL coefficient (leads to reward hacking)
- [ ] Not using gradient clipping (PPO can produce large gradients)

---

**Next**: Tutorial 41 - Advanced Preference Optimization (DPO variants, GRPO, and beyond)